In [13]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
print("All good! 🚀")



All good! 🚀


In [14]:
url = "https://data.cityofnewyork.us/resource/4b4i-vvec.json"

params = {
    "$select": "tpep_pickup_datetime, tip_amount, fare_amount, trip_distance, passenger_count, payment_type",
    "$where": "payment_type = '1' AND tip_amount > 0 AND tpep_pickup_datetime >= '2023-01-01T00:00:00' AND tpep_pickup_datetime < '2023-02-01T00:00:00'",
    "$limit": 500000,
    "$order": "tpep_pickup_datetime ASC"
}

response = requests.get(url, params=params)
df = pd.DataFrame(response.json())

print(f"Rows retrieved: {len(df)}")
print(df.head())

Rows retrieved: 500000
      tpep_pickup_datetime tip_amount fare_amount trip_distance  \
0  2023-01-01T00:00:09.000       7.44        19.8           3.8   
1  2023-01-01T00:00:13.000       25.0        34.5          8.97   
2  2023-01-01T00:00:18.000        4.9        11.4           2.1   
3  2023-01-01T00:00:47.000        5.0         6.5          0.95   
4  2023-01-01T00:00:57.000        5.0        38.0          9.98   

  passenger_count payment_type  
0             1.0            1  
1             1.0            1  
2             1.0            1  
3             3.0            1  
4             1.0            1  


In [15]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   tpep_pickup_datetime  500000 non-null  str  
 1   tip_amount            500000 non-null  str  
 2   fare_amount           500000 non-null  str  
 3   trip_distance         500000 non-null  str  
 4   passenger_count       500000 non-null  str  
 5   payment_type          500000 non-null  str  
dtypes: str(6)
memory usage: 22.9 MB


In [16]:
df.head(100).to_csv("/Users/manostsili/Desktop/sample.csv", index=False)
print("Saved to Desktop ✅")

Saved to Desktop ✅


In [17]:
import requests

# NYC coordinates (Central Park)
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 40.7831,
    "longitude": -73.9712,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m,weathercode",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data = response.json()

# Build dataframe from the response
weather_df = pd.DataFrame({
    "date_hour": pd.to_datetime(data["hourly"]["time"]),
    "temperature": data["hourly"]["temperature_2m"],
    "precipitation": data["hourly"]["precipitation"],
    "rain": data["hourly"]["rain"],
    "snowfall": data["hourly"]["snowfall"],
    "windspeed": data["hourly"]["windspeed_10m"],
    "weathercode": data["hourly"]["weathercode"],
})

print(f"Weather rows: {len(weather_df)}")
print(weather_df.head(10))

Weather rows: 8760
            date_hour  temperature  precipitation  rain  snowfall  windspeed  \
0 2023-01-01 00:00:00         10.9            1.1   1.1       0.0       11.7   
1 2023-01-01 01:00:00         10.9            0.9   0.9       0.0        9.1   
2 2023-01-01 02:00:00         10.8            0.1   0.1       0.0       14.4   
3 2023-01-01 03:00:00         10.8            0.0   0.0       0.0       16.6   
4 2023-01-01 04:00:00         10.1            0.0   0.0       0.0       16.9   
5 2023-01-01 05:00:00          9.4            0.0   0.0       0.0       17.2   
6 2023-01-01 06:00:00          8.9            0.0   0.0       0.0       16.9   
7 2023-01-01 07:00:00          8.3            0.0   0.0       0.0       14.9   
8 2023-01-01 08:00:00          8.2            0.0   0.0       0.0       12.3   
9 2023-01-01 09:00:00          8.4            0.0   0.0       0.0       12.6   

   weathercode  
0           55  
1           53  
2           51  
3            0  
4            0 

In [18]:
weather_df.head(743).to_csv("/Users/manostsili/Desktop/sample_weather.csv", index=False)
print("Saved to Desktop ✅")

Saved to Desktop ✅


In [ ]:
# Fix types
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])  # convert first!
df["tip_amount"] = pd.to_numeric(df["tip_amount"])
df["fare_amount"] = pd.to_numeric(df["fare_amount"])
df["trip_distance"] = pd.to_numeric(df["trip_distance"])

# Remove outliers
df = df[df["fare_amount"] >= 2.50]
df = df[df["trip_distance"] > 0]

# Calculate tip percentage
df["tip_pct"] = (df["tip_amount"] / df["fare_amount"]) * 100
df = df[df["tip_pct"] <= 100]

# Extract time features
df["hour"] = df["tpep_pickup_datetime"].dt.hour
df["day_of_week"] = df["tpep_pickup_datetime"].dt.day_name()
df["is_late_night"] = df["hour"].between(22, 23) | df["hour"].between(0, 4)
df["date_hour"] = df["tpep_pickup_datetime"].dt.floor("h")  # now this will work

print(f"Clean rows: {len(df)}")
print(df[["tip_amount", "fare_amount", "tip_pct", "hour", "is_late_night"]].describe())

AttributeError: Can only use .dt accessor with datetimelike values